# Fase 1 — Predicción de retrasos en la llegada de vuelos

Modelos y Simulación de Sistemas I — 2026-II · Grupo B


## 0. Configuración


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Semilla global del proyecto: garantiza reproducibilidad en splits y estimadores
RANDOM_STATE = 42


In [ ]:
# Ruta relativa
df = pd.read_csv("data/flight_data_2024_sample.csv")


## 1. Introducción


### 1.1 Descripción del problema

El transporte aéreo comercial funciona como una red fuertemente acoplada: cada aeronave
encadena varios vuelos a lo largo del día, las tripulaciones tienen jornadas reguladas y
los aeropuertos operan con franjas horarias asignadas con poco margen. En un sistema así,
un retraso rara vez se queda donde nace. Un vuelo que sale tarde de su primer origen
arrastra el desfase a los tramos siguientes de la misma aeronave, puede dejar a una
tripulación fuera de su límite legal de servicio y hace que pasajeros con conexión pierdan
su enlace. Lo que empieza como una demora local se propaga por la red durante el resto de
la jornada.

Las consecuencias son a la vez logísticas y económicas. Para la aerolínea, un retraso
significa combustible adicional, reacomodación de pasajeros, compensaciones, personal en
sobretiempo y, sobre todo, una programación que hay que rehacer sobre la marcha. Para los
aeropuertos implica congestión de posiciones y de personal en tierra en momentos que no
estaban previstos. Y para el pasajero se traduce en tiempo perdido, conexiones falladas y
planes rotos. Por eso los organismos reguladores estadounidenses —la FAA y el Bureau of
Transportation Statistics— llevan décadas midiendo y publicando la puntualidad de cada
vuelo doméstico: es un indicador de desempeño del sector, no un dato anecdótico.

Frente a este panorama, poder **anticipar** el retraso de un vuelo antes de que despegue
tiene un valor operativo claro. Permite avisar a los pasajeros con tiempo, replanificar
conexiones ajustadas, redistribuir recursos en tierra hacia los vuelos que probablemente
los necesitarán y tomar decisiones de programación con una expectativa cuantificada en
lugar de con una intuición. Ese es el problema que aborda este trabajo.


### 1.2 Objetivo del modelo

El objetivo es construir un modelo capaz de estimar cuántos minutos de retraso acumulará
un vuelo en su llegada, utilizando **exclusivamente información conocida antes del
despegue**: la aerolínea operadora, los aeropuertos de origen y destino, los horarios
programados de salida y llegada, la duración y la distancia previstas, y la posición del
vuelo en el calendario (mes y día de la semana).

Esa restricción es deliberada y es, de hecho, lo que decide si el modelo sirve para algo.
El conjunto de datos incluye campos que describen el vuelo con enorme precisión —el
retraso en la salida, los tiempos de rodaje, las horas reales de despegue y aterrizaje—
pero todos ellos se conocen *durante o después* del vuelo. Un modelo que los utilizara
alcanzaría un desempeño excelente en la evaluación y sería inútil en producción, porque en
el momento en que la predicción tendría valor —horas antes del despegue, cuando todavía se
puede reprogramar una conexión o avisar a un pasajero— esos datos sencillamente no
existen. Usarlos constituiría **fuga de información**, y evitarla es la restricción central
de este trabajo.

Dicho de otro modo: preferimos un modelo honesto y menos preciso, cuyas predicciones puedan
producirse en el instante en que se necesitan, antes que un modelo aparentemente
sobresaliente que solo funciona cuando la respuesta ya se conoce. Las variables excluidas
se enumeran y se justifican una por una en la sección 4, y las medidas concretas adoptadas
para prevenir la fuga se documentan en la sección 6.


### 1.3 Tipo de problema

El trabajo se plantea como un problema de **regresión supervisada**. Es supervisado porque
cada vuelo histórico viene acompañado de su retraso real observado, que actúa como
etiqueta con la que el modelo aprende; y es de regresión porque la cantidad que queremos
predecir, `arr_delay`, es continua y se mide en minutos.

Existía una alternativa razonable: convertir el problema en una **clasificación binaria**
definiendo `is_delayed = (arr_delay > 15).astype(int)`, donde 15 minutos es el umbral con
el que la FAA y el BTS consideran oficialmente que un vuelo llegó tarde. Esa formulación
es más sencilla de evaluar y suele producir métricas más vistosas.

Optamos por la regresión por dos razones. La primera es informativa: la clasificación
binaria descarta la magnitud del retraso, y para las decisiones que motivan el modelo esa
magnitud es justamente lo que importa — un retraso de 20 minutos y uno de tres horas caen
en la misma clase pero tienen consecuencias operativas incomparables. La segunda es
metodológica: la salida continua es más general, ya que a partir de una predicción en
minutos siempre se puede derivar la clasificación aplicando el umbral, mientras que el
camino inverso no existe. Asumimos a cambio un problema más exigente, con métricas que
previsiblemente serán modestas; ese resultado se interpreta con honestidad en la
sección 10 en lugar de esquivarse.


### 1.4 Variable objetivo

La variable objetivo es **`arr_delay`**: la diferencia, **en minutos**, entre la hora real
de llegada del vuelo y la hora de llegada programada en el itinerario.

Su signo tiene significado propio y conviene tenerlo presente desde el principio: un valor
positivo indica que el vuelo llegó tarde, un valor negativo indica que llegó adelantado —
algo frecuente, porque las aerolíneas incorporan holgura en los tiempos de bloque
publicados — y un valor cercano a cero indica que llegó en hora. Por eso el objetivo no es
una magnitud estrictamente positiva, y cualquier transformación que asuma lo contrario
(un logaritmo, por ejemplo) requeriría un tratamiento previo del rango negativo.

La distribución de `arr_delay`, su asimetría y sus valores atípicos se examinan con datos
en la sección 3.


### 1.5 Fuente de los datos

Los datos provienen del conjunto **Flight Delay Dataset 2024**, publicado por Hrishit Patil
en Kaggle:

> https://www.kaggle.com/datasets/hrishitpatil/flight-data-2024

Ese conjunto no es una recolección propia del autor, sino una consolidación de los
registros públicos del **Bureau of Transportation Statistics (BTS)** del Departamento de
Transporte de los Estados Unidos, concretamente de la base *TranStats On-Time
Performance*, en la que las aerolíneas comerciales están obligadas por regulación a
reportar mes a mes los horarios programados y reales de cada vuelo doméstico. Que el
origen sea un registro regulatorio obligatorio, y no una muestra voluntaria, es una
garantía razonable de cobertura y de consistencia en la definición de los campos.

Por su volumen, el archivo completo no se versiona en este repositorio. El trabajo se
realiza sobre la muestra `data/flight_data_2024_sample.csv`, cuyas dimensiones exactas se
verifican con código en la sección 2. Las instrucciones para obtener los datos se
documentan en el `README.md` de esta fase.


## 2. Descripción del conjunto de datos


## 3. Exploración de datos (EDA)


## 4. Preparación de los datos


## 5. Separación train / test


## 6. Prevención de fuga de información


## 7. Modelo base (baseline)


## 8. Modelo predictivo


## 9. Evaluación


## 10. Interpretación de resultados


## 11. Serialización y verificación
